# Publication Baseline Colab Runner

Purpose: prepare a safe, reproducible manual Google Colab workflow for the NS-MC-GAN publication baseline configs. This notebook is intentionally conservative: it validates the repository, discovers the real training entrypoint, writes logs and manifests, and defaults to smoke mode with no training.

Safety boundary: use this only in your own authenticated Colab session. Do not paste private tokens, do not hard-code secrets, do not use keep-alive hacks, and do not claim dry-run or smoke output as full experiment results.


## Source Setup

Supported `SOURCE_MODE` values are `drive_zip`, `upload_zip`, and `git_clone`.

- `drive_zip`: mount Google Drive and extract a repo zip you placed there.
- `upload_zip`: upload a local zip through `files.upload()`.
- `git_clone`: clone a public Git URL. Private credentials are intentionally unsupported here.


In [ ]:
# User Configuration
SOURCE_MODE = "upload_zip"  # drive_zip, upload_zip, or git_clone
DRIVE_ZIP_PATH = "/content/drive/MyDrive/ns_mc_gan_gi_code_pub_colab.zip"
GIT_URL = ""
GIT_BRANCH = ""
REPO_DIR = "/content/ns_mc_gan_gi_code"
NOTEBOOK_FILENAME = "colab/pub_baselines_colab_runner.ipynb"

RUN_MODE = "smoke"  # smoke, single, or matrix
DRY_RUN = True
MAX_CONFIGS = 1  # keep single dry-run to exactly one config; set None only after smoke/single pass
SINGLE_CONFIG = ""  # empty means: use the first valid configs/pub_baselines/colab config
MATRIX_CONFIGS = [
    "configs/pub_baselines/colab/unet_rad5_pub_colab.yaml",
    "configs/pub_baselines/colab/unet_scr5_pub_colab.yaml",
    "configs/pub_baselines/colab/unrolled_ista_rad5_pub_colab.yaml",
    "configs/pub_baselines/colab/unrolled_ista_scr5_pub_colab.yaml",
    "configs/pub_baselines/colab/resunet_rad5_pub_colab.yaml",
    "configs/pub_baselines/colab/resunet_scr5_pub_colab.yaml",
]

OUTPUT_ROOT = "/content/pub_baseline_colab_runs"
INSTALL_DEPENDENCIES = False
TRAIN_EXTRA_ARGS = []
DEVICE = "cuda"


In [ ]:
from __future__ import annotations

import csv
import json
import os
import shlex
import shutil
import subprocess
import sys
import textwrap
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%SZ")
RUN_DIR = Path(OUTPUT_ROOT).expanduser() / f"pub_baselines_{RUN_STAMP}_{RUN_MODE}"
LOG_DIR = RUN_DIR / "logs"
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

COMMAND_LOG: list[dict[str, Any]] = []
RETURN_SUMMARY: list[dict[str, Any]] = []
REPO_ROOT = Path(REPO_DIR).expanduser()
TRACEBACK_TEXT = "none"


def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=False) + "\n", encoding="utf-8")


def write_csv(path: Path, rows: list[dict[str, Any]], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def command_to_shell(cmd: list[str]) -> str:
    return shlex.join([str(part) for part in cmd])


def run_command(
    cmd: list[str],
    cwd: Path | None = None,
    check: bool = False,
    dry_run: bool = False,
    label: str | None = None,
    config_path: str | None = None,
    command_kind: str = "utility",
    reason_not_executed: str | None = None,
) -> subprocess.CompletedProcess[str] | None:
    cwd_text = str(cwd or Path.cwd())
    command = [str(part) for part in cmd]
    shell_command = command_to_shell(command)
    started = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
    record: dict[str, Any] = {
        "label": label or "command",
        "config_path": config_path,
        "command": command,
        "cmd": command,
        "shell_command": shell_command,
        "cwd": cwd_text,
        "started_utc": started,
        "dry_run": bool(dry_run),
        "run_mode": RUN_MODE,
        "command_kind": command_kind,
        "status": "planned",
        "returncode": None,
        "reason_not_executed": None,
    }
    print(f"$ ({cwd_text}) {shell_command}")
    if dry_run:
        reason = reason_not_executed or "DRY_RUN=True; command was logged but not executed."
        print(f"DRY_RUN: {reason}")
        record["status"] = "dry_run"
        record["returncode"] = "dry_run"
        record["reason_not_executed"] = reason
        COMMAND_LOG.append(record)
        RETURN_SUMMARY.append(
            {
                "label": record["label"],
                "status": "dry_run",
                "returncode": "dry_run",
                "config_path": config_path,
                "run_mode": RUN_MODE,
            }
        )
        return None
    result = subprocess.run(command, cwd=cwd_text, text=True, capture_output=True)
    record["returncode"] = result.returncode
    record["stdout_tail"] = result.stdout[-4000:]
    record["stderr_tail"] = result.stderr[-4000:]
    record["status"] = "succeeded" if result.returncode == 0 else "failed"
    COMMAND_LOG.append(record)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-4000:])
    RETURN_SUMMARY.append(
        {
            "label": record["label"],
            "status": record["status"],
            "returncode": result.returncode,
            "config_path": config_path,
            "run_mode": RUN_MODE,
        }
    )
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}: {shell_command}")
    return result


def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.rglob("requirements.txt")]
    for candidate in candidates:
        root = candidate if candidate.is_dir() else candidate.parent
        if (root / "src").is_dir() and (root / "configs").is_dir():
            return root.resolve()
    raise FileNotFoundError(f"Could not find repo root under {start}")


def extract_zip(zip_path: Path, destination: Path) -> Path:
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(zip_path), str(destination))
    return find_repo_root(destination)


In [ ]:
# Source Setup
mode = SOURCE_MODE.strip().lower()
if mode == "drive_zip":
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    REPO_ROOT = extract_zip(Path(DRIVE_ZIP_PATH), Path(REPO_DIR).with_name("repo_from_drive_zip"))
elif mode == "upload_zip":
    from google.colab import files  # type: ignore

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No zip was uploaded.")
    zip_name = next(iter(uploaded))
    REPO_ROOT = extract_zip(Path(zip_name), Path(REPO_DIR).with_name("repo_from_upload_zip"))
elif mode == "git_clone":
    if not GIT_URL:
        raise ValueError("Set GIT_URL to a public repository URL before using git_clone.")
    target = Path(REPO_DIR)
    if target.exists():
        shutil.rmtree(target)
    cmd = ["git", "clone", "--depth", "1"]
    if GIT_BRANCH:
        cmd.extend(["--branch", GIT_BRANCH])
    cmd.extend([GIT_URL, str(target)])
    run_command(cmd, cwd=Path("/content"), check=True, label="git_clone")
    REPO_ROOT = find_repo_root(target)
else:
    raise ValueError(f"Unknown SOURCE_MODE: {SOURCE_MODE!r}")

os.chdir(REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"Run directory: {RUN_DIR}")


## Environment Check

This cell records Python, working directory, GPU visibility, optional `nvidia-smi`, optional torch CUDA status, and disk space.


In [ ]:
# Environment Check
env_log: dict[str, Any] = {
    "python": sys.version,
    "python_executable": sys.executable,
    "working_directory": str(Path.cwd()),
    "repo_root": str(REPO_ROOT),
    "run_dir": str(RUN_DIR),
    "time_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
}
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Working directory:", Path.cwd())

nvidia = shutil.which("nvidia-smi")
if nvidia:
    result = subprocess.run([nvidia], text=True, capture_output=True)
    env_log["nvidia_smi_returncode"] = result.returncode
    env_log["nvidia_smi_stdout_tail"] = result.stdout[-4000:]
    env_log["nvidia_smi_stderr_tail"] = result.stderr[-4000:]
    print(result.stdout[-4000:] or result.stderr[-4000:])
else:
    env_log["nvidia_smi"] = "not_found"
    print("nvidia-smi not found; continuing without it.")

try:
    import torch

    env_log["torch_version"] = torch.__version__
    env_log["torch_cuda_available"] = bool(torch.cuda.is_available())
    env_log["torch_cuda_device_count"] = int(torch.cuda.device_count())
    print("torch:", torch.__version__)
    print("torch.cuda.is_available():", torch.cuda.is_available())
except Exception as exc:
    env_log["torch_import_error"] = repr(exc)
    print("torch import failed or torch is not installed:", repr(exc))

usage = shutil.disk_usage(str(Path.cwd()))
env_log["disk_usage_bytes"] = {"total": usage.total, "used": usage.used, "free": usage.free}
print("Disk free GB:", round(usage.free / (1024 ** 3), 2))
write_json(LOG_DIR / "environment_log.json", env_log)


## Dependency Installation

The notebook inspects dependency files from the repo. It only installs from existing repo requirement files when `INSTALL_DEPENDENCIES = True`.


In [ ]:
# Dependency Installation
dependency_files = [
    "requirements.txt",
    "requirements-colab.txt",
    "environment.yml",
    "environment.yaml",
    "pyproject.toml",
    "setup.py",
]
found_dependency_files = []
for rel in dependency_files:
    path = REPO_ROOT / rel
    if path.exists():
        found_dependency_files.append(rel)
        print(f"--- {rel} ---")
        print(path.read_text(encoding="utf-8", errors="replace")[:4000])
write_json(LOG_DIR / "dependency_files.json", found_dependency_files)

requirements_path = REPO_ROOT / "requirements.txt"
if INSTALL_DEPENDENCIES:
    if requirements_path.exists():
        run_command([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)], cwd=REPO_ROOT, check=True, label="pip_install_requirements")
    else:
        print("INSTALL_DEPENDENCIES=True, but requirements.txt was not found.")
else:
    print("INSTALL_DEPENDENCIES=False; no packages installed by this notebook.")


## Config Validation

This uses the repository scripts to render Colab baseline configs, validate `configs/pub_baselines`, and list the config set that can be used by run modes.


In [ ]:
# Config Validation
render_script = REPO_ROOT / "scripts" / "render_pub_baseline_colab_configs.py"
validate_script = REPO_ROOT / "scripts" / "validate_pub_baseline_configs.py"
if render_script.exists():
    run_command([sys.executable, str(render_script)], cwd=REPO_ROOT, check=True, label="render_pub_baseline_colab_configs")
else:
    print("render_pub_baseline_colab_configs.py not found; skipping render step.")

if not validate_script.exists():
    raise FileNotFoundError(validate_script)
run_command([sys.executable, str(validate_script)], cwd=REPO_ROOT, check=True, label="validate_pub_baseline_configs")

config_root = REPO_ROOT / "configs" / "pub_baselines"
colab_config_root = config_root / "colab"
if not config_root.exists():
    raise FileNotFoundError(config_root)
config_paths = sorted(colab_config_root.glob("*.yaml")) if colab_config_root.exists() else sorted(config_root.glob("*.yaml"))
config_list = [str(path.relative_to(REPO_ROOT).as_posix()) for path in config_paths]
print("Config count:", len(config_list))
for item in config_list:
    print("-", item)
write_json(LOG_DIR / "config_list.json", config_list)


## Baseline Command Discovery

The runner checks the actual repository for a training entrypoint. It writes a dry-run command manifest and records any ambiguity instead of inventing a fake training command.


In [ ]:
# Baseline Command Discovery
train_py = REPO_ROOT / "src" / "train.py"
eval_auto_py = REPO_ROOT / "src" / "eval_auto.py"
script_hits: list[str] = []
for folder in [REPO_ROOT / "scripts", REPO_ROOT / "src", REPO_ROOT]:
    if not folder.exists():
        continue
    for path in folder.rglob("*"):
        if path.suffix.lower() not in {".py", ".sh", ".ps1", ".md"}:
            continue
        try:
            text = path.read_text(encoding="utf-8", errors="replace")
        except OSError:
            continue
        if "src.train" in text or "python -m src.train" in text:
            script_hits.append(str(path.relative_to(REPO_ROOT).as_posix()))

train_text = train_py.read_text(encoding="utf-8", errors="replace") if train_py.exists() else ""
eval_auto_text = eval_auto_py.read_text(encoding="utf-8", errors="replace") if eval_auto_py.exists() else ""
entrypoint_ok = train_py.exists() and "ArgumentParser" in train_text and "--config" in train_text
_eval_entrypoint_ok = eval_auto_py.exists() and "ArgumentParser" in eval_auto_text and "--config" in eval_auto_text
command_manifest: dict[str, Any] = {
    "selected_entrypoint": "src.train" if entrypoint_ok else None,
    "selected_command_kind": "training",
    "command_template": [sys.executable, "-m", "src.train", "--config", "{config}", "--device", DEVICE],
    "candidate_commands": [
        {
            "kind": "training",
            "entrypoint": "src.train",
            "template": [sys.executable, "-m", "src.train", "--config", "{config}", "--device", DEVICE],
            "confirmed": bool(entrypoint_ok),
        },
        {
            "kind": "evaluation_after_training",
            "entrypoint": "src.eval_auto",
            "template": [sys.executable, "-m", "src.eval_auto", "--output_dir", "{output_dir}", "--config", "{config}", "--device", DEVICE],
            "confirmed": bool(_eval_entrypoint_ok),
            "note": "Requires a completed training output_dir, so it is not scheduled for the first single dry-run.",
        },
    ],
    "evidence_files": sorted(set(script_hits))[:50],
    "ambiguous": not entrypoint_ok,
    "ambiguity_note": "src.train with --config was found." if entrypoint_ok else "Could not confirm src.train --config; keep commands as dry-run only.",
}
print(json.dumps(command_manifest, indent=2))
write_json(LOG_DIR / "command_discovery_manifest.json", command_manifest)


## Run Modes

- `smoke`: default; validate configs and write manifests only.
- `single`: prepare or run one selected config.
- `matrix`: prepare or run multiple configs; not default.


In [ ]:
# Run Modes
mode = RUN_MODE.strip().lower()
max_configs = None if MAX_CONFIGS is None else max(1, int(MAX_CONFIGS))
selected_config: str | None = None


def first_available_config() -> str:
    if not config_list:
        raise RuntimeError("No Colab publication baseline configs were found.")
    return config_list[0]


def choose_single_config() -> str:
    requested = str(SINGLE_CONFIG or "").strip()
    if requested:
        if (REPO_ROOT / requested).exists():
            return requested
        raise FileNotFoundError(f"SINGLE_CONFIG was set but does not exist: {requested}")
    return first_available_config()

if mode == "smoke":
    selected_configs: list[str] = []
elif mode == "single":
    selected_config = choose_single_config()
    selected_configs = [selected_config]
elif mode == "matrix":
    matrix_candidates = list(MATRIX_CONFIGS or config_list)
    selected_configs = matrix_candidates[:max_configs] if max_configs is not None else matrix_candidates
    selected_config = selected_configs[0] if selected_configs else None
else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE!r}")

if mode == "single" and max_configs is not None:
    selected_configs = selected_configs[:1]
if mode == "matrix" and max_configs is not None:
    selected_configs = selected_configs[:max_configs]

missing = [cfg for cfg in selected_configs if not (REPO_ROOT / cfg).exists()]
if missing:
    raise FileNotFoundError(f"Selected config(s) not found: {missing}")

planned_commands: list[dict[str, Any]] = []
for cfg in selected_configs:
    cmd = [sys.executable, "-m", "src.train", "--config", cfg, "--device", DEVICE, *TRAIN_EXTRA_ARGS]
    planned_commands.append(
        {
            "config": cfg,
            "command": cmd,
            "cmd": cmd,
            "shell_command": command_to_shell(cmd),
            "command_kind": "training",
            "entrypoint": "src.train",
        }
    )

write_json(LOG_DIR / "planned_training_commands.json", planned_commands)
print("configs_seen:", len(config_list))
print("selected_config:", selected_config)
print("Planned training commands:", len(planned_commands))
for item in planned_commands:
    print(item["shell_command"])

if mode == "smoke":
    print("Smoke mode: no training commands are executed.")
elif command_manifest.get("ambiguous"):
    print("Training entrypoint is ambiguous; commands are logged as dry-run only.")
    for item in planned_commands:
        run_command(
            item["command"],
            cwd=REPO_ROOT,
            dry_run=True,
            label=f"train_{Path(item['config']).stem}",
            config_path=item["config"],
            command_kind="training",
            reason_not_executed="Training entrypoint ambiguity; command logged for inspection only.",
        )
else:
    for item in planned_commands:
        run_command(
            item["command"],
            cwd=REPO_ROOT,
            dry_run=DRY_RUN,
            label=f"train_{Path(item['config']).stem}",
            config_path=item["config"],
            command_kind="training",
            reason_not_executed="DRY_RUN=True; inspect this exact command before running real training.",
        )


## Logging And Artifacts

This cell writes the command log, return-code summary, artifact manifest JSON/CSV, and a zip bundle when possible.


In [ ]:
# Logging And Artifacts
def command_rows(kind: str | None = None) -> list[dict[str, Any]]:
    rows = COMMAND_LOG
    if kind is not None:
        rows = [row for row in rows if row.get("command_kind") == kind]
    return rows


def build_return_code_summary() -> dict[str, Any]:
    training_rows = command_rows("training")
    attempted_rows = [row for row in training_rows if row.get("status") in {"succeeded", "failed"}]
    succeeded_rows = [row for row in attempted_rows if row.get("status") == "succeeded"]
    failed_rows = [row for row in attempted_rows if row.get("status") == "failed"]
    dry_run_rows = [row for row in training_rows if row.get("status") == "dry_run"]
    first_command = training_rows[0] if training_rows else None
    return {
        "run_mode": RUN_MODE,
        "dry_run": bool(DRY_RUN),
        "source_mode": SOURCE_MODE,
        "configs_seen": len(config_list) if "config_list" in globals() else None,
        "planned_training_commands": len(planned_commands) if "planned_commands" in globals() else 0,
        "dry_run_commands": len(dry_run_rows),
        "training_attempted": len(attempted_rows),
        "training_succeeded": len(succeeded_rows),
        "training_failed": len(failed_rows),
        "selected_config": selected_config,
        "unresolved_ambiguity": bool(command_manifest.get("ambiguous")) if "command_manifest" in globals() else None,
        "ambiguity_note": command_manifest.get("ambiguity_note") if "command_manifest" in globals() else None,
        "first_command_log_entry": first_command,
    }

return_code_summary = build_return_code_summary()
write_json(LOG_DIR / "command_log.json", COMMAND_LOG)
write_json(LOG_DIR / "return_code_summary.json", return_code_summary)
summary_csv_row = {
    key: json.dumps(value, sort_keys=True) if isinstance(value, (dict, list)) else value
    for key, value in return_code_summary.items()
}
write_csv(
    LOG_DIR / "return_code_summary.csv",
    [summary_csv_row],
    list(summary_csv_row.keys()),
)

collector = REPO_ROOT / "scripts" / "collect_colab_artifacts.py"
if collector.exists():
    collector_result = subprocess.run(
        [sys.executable, str(collector), str(RUN_DIR)],
        cwd=str(REPO_ROOT),
        text=True,
        capture_output=True,
    )
    if collector_result.stdout:
        print(collector_result.stdout[-4000:])
    if collector_result.stderr:
        print(collector_result.stderr[-4000:])
    if collector_result.returncode != 0:
        print(f"collect_colab_artifacts.py returned {collector_result.returncode}; continuing to summary.")
else:
    print("collect_colab_artifacts.py not found; writing command logs only.")

try:
    archive_path = shutil.make_archive(str(RUN_DIR), "zip", root_dir=str(RUN_DIR))
    print("Zip bundle:", archive_path)
except Exception as exc:
    archive_path = None
    print("Could not create zip bundle:", repr(exc))


## End-of-run Summary

The printed block named `COLAB_SMOKE_COPY_TO_CHATGPT` is intended to be copied back into ChatGPT after the first smoke test.


In [ ]:
# End-of-run Summary
return_code_summary = build_return_code_summary()
training_rows = command_rows("training")
attempted = [row for row in training_rows if row.get("status") in {"succeeded", "failed"}]
succeeded = [row for row in attempted if row.get("status") == "succeeded"]
failed = [row for row in attempted if row.get("status") == "failed"]
dry_run_rows = [row for row in training_rows if row.get("status") == "dry_run"]
first_command_entry = training_rows[0] if training_rows else None
first_command_text = first_command_entry.get("shell_command") if first_command_entry else "none"
runtime_summary = (
    f"python={env_log.get('python_executable', sys.executable)}; "
    f"torch_cuda_available={env_log.get('torch_cuda_available', 'unknown')}; "
    f"disk_free_gb={round((env_log.get('disk_usage_bytes') or {}).get('free', 0) / (1024 ** 3), 2)}"
)

section_list = [
    "Title and purpose",
    "User configuration cell",
    "Source setup",
    "Environment check",
    "Dependency installation",
    "Config validation",
    "Baseline command discovery",
    "Run modes",
    "Logging and artifacts",
    "End-of-run summary",
    "Completion notification",
]

if RUN_MODE.strip().lower() == "single" and DRY_RUN:
    block_label = "COLAB_SINGLE_DRYRUN_COPY_TO_CHATGPT"
elif RUN_MODE.strip().lower() == "smoke":
    block_label = "COLAB_SMOKE_COPY_TO_CHATGPT"
else:
    block_label = "COLAB_RUN_COPY_TO_CHATGPT"

copy_block = f"""
{block_label}
notebook_filename: {NOTEBOOK_FILENAME}
runtime_summary: {runtime_summary}
source_mode: {SOURCE_MODE}
run_mode: {RUN_MODE}
dry_run: {DRY_RUN}
repo_root: {REPO_ROOT}
run_dir: {RUN_DIR}
configs_seen: {return_code_summary.get('configs_seen')}
selected_config: {selected_config}
planned_training_commands: {return_code_summary.get('planned_training_commands')}
dry_run_commands: {len(dry_run_rows)}
training_attempted: {len(attempted)}
training_succeeded: {len(succeeded)}
training_failed: {len(failed)}
command_log: {LOG_DIR / 'command_log.json'}
return_code_summary: {LOG_DIR / 'return_code_summary.json'}
artifact_manifest_json: {RUN_DIR / 'artifact_manifest.json'}
artifact_manifest_csv: {RUN_DIR / 'artifact_manifest.csv'}
zip_bundle: {archive_path}
exact_planned_command: {first_command_text}
first_command_log_entry: {json.dumps(first_command_entry, sort_keys=True) if first_command_entry else 'none'}
traceback: {TRACEBACK_TEXT}
sections: {', '.join(section_list)}
next_steps: For single dry-run, inspect command_log.json and return_code_summary.json. Only set DRY_RUN=False for one real single run after the command is correct.
""".strip()

print(copy_block)
summary_path = LOG_DIR / f"{block_label}.txt"
summary_path.write_text(copy_block + "\n", encoding="utf-8")
print("Summary written to:", summary_path)


## Completion Notification

This emits a harmless text notification and terminal bell where supported.


In [ ]:
# Completion Notification
print("\a")
print("Colab runner finished. Review COLAB_SMOKE_COPY_TO_CHATGPT above before starting any real training.")
